In [ ]:
# Install required packages
!pip install stable-baselines3[extra] gymnasium matplotlib pygame

zsh:1: no matches found: stable-baselines3[extra]


In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
import time

In [ ]:
env = gym.make('LunarLander-v3')

model = PPO(
    policy = "MlpPolicy",
    env = env,
    verbose = 1,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.99,
    learning_rate = 2.5e-4,
    ent_coef = 0.01,
    vf_coef = 0.5,
    max_grad_norm = 0.5,
    device = "mps"
)

/Users/ryanchen/.virtualenvs/.venv/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Using mps device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/Users/ryanchen/.virtualenvs/.venv/lib/python3.13/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [ ]:
model.learn(total_timesteps=1E6)
model_name = "RyanModelV2"
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 93.8     |
|    ep_rew_mean     | -212     |
| time/              |          |
|    fps             | 318      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 1024     |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 98.5          |
|    ep_rew_mean          | -217          |
| time/                   |               |
|    fps                  | 306           |
|    iterations           | 2             |
|    time_elapsed         | 6             |
|    total_timesteps      | 2048          |
| train/                  |               |
|    approx_kl            | 0.00089399004 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.39         |
|    explained_variance   | 0.00282       |


In [4]:
# Load your trained PPO model
model = PPO.load("RyanModelV2")  # Load from the extracted policy.pth

print("Model loaded successfully!")
print(f"Model action space: {model.action_space}")
print(f"Model observation space: {model.observation_space}")

Model loaded successfully!
Model action space: Discrete(4)
Model observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)


In [ ]:
env = gym.make('LunarLander-v3', render_mode='human')

def visualize_agent_realtime(episodes=3, delay=0.01):
    """Visualize the agent playing in real-time"""
    print(f"Starting visualization with environment: {env.spec.id}")
    
    for episode in range(episodes):
        obs, info = env.reset()
        done = False
        truncated = False
        total_reward = 0
        step_count = 0
        
        print(f"\n=== Episode {episode + 1} ===")
        
        while not done and not truncated:
            # Get action from the trained model
            action, _ = model.predict(obs, deterministic=True)
            
            # Take action in environment
            obs, reward, done, truncated, info = env.step(action)
            total_reward += reward
            step_count += 1
            
            # Render the environment
            try:
                env.render()
            except Exception as e:
                print(f"Render error: {e}")
                break
            
            # Slow down for viewing
            time.sleep(delay)
            
            # Safety limit to prevent infinite loops
            if step_count > 1000:
                print("Step limit reached, ending episode")
                break
            
        print(f"Episode {episode + 1} finished with reward: {total_reward}, steps: {step_count}")
    
    env.close()
    print("Visualization complete!")

# Run the visualization now
visualize_agent_realtime(episodes=2, delay=0.05)

Starting visualization with environment: LunarLander-v3

=== Episode 1 ===
Episode 1 finished with reward: 249.073714659902, steps: 335

=== Episode 2 ===
Episode 2 finished with reward: 202.1365374715788, steps: 361
Visualization complete!


: 

### Q-learning

In [1]:
import gymnasium as gym 
import random 
import numpy as np 

In [2]:
env = gym.make("FrozenLake-v1", map_name = "4x4", is_slippery = False, render_mode="rgb_array")

In [3]:
env.observation_space

Discrete(16)

In [4]:
env.action_space

Discrete(4)

In [5]:
state_space = env.observation_space.n
action_space = env.action_space.n
print(f"State space: {state_space}")
print(f"Action space: {action_space}")

State space: 16
Action space: 4


In [8]:
def q_table(state_space, action_space):
    return np.zeros((state_space, action_space))

q = q_table(state_space, action_space)

In [9]:
def greedy_policy(state, q_table):
    '''
    state: current state
    q_table: Q-table
    returns action with highest Q-value for the given state
    '''
    action = np.argmax(q_table[state][:])
    return action

In [10]:
import random 

def epsilon_greedy(epislon, Q, state):
    if random.random() < epislon:
        return random.randint(0, 3)
    else:
        return greedy_policy(state, Q)
    

In [18]:
def update_q_table(Q, state, action, reward, next_state, gamma, lr):
    Q[state][action] += lr * (reward + gamma * np.max(Q[next_state]) - Q[state][action])
    return Q

In [34]:
n_training_steps = 100000
lr = 0.2
n_evaluations = 100
max_steps = 99 
gamma = 0.95 
epsilon_decay = 0.999
epsilon = 1.0

In [ ]:
from tqdm.auto import tqdm

def train_loop(episodes, lr, gamma, max_steps, epsilon, epsilon_decay, q_table):
    for episode in tqdm(range(episodes)):
        state, info = env.reset()
        
        for step in range(max_steps):
            action = epsilon_greedy(epsilon, q_table, state)
            next_state, reward, done, truncated, info = env.step(action) 
            
            q_table = update_q_table(q_table, state, action, reward, next_state, gamma, lr)
            state = next_state
            if step % 10 == 0:
                print(f"Episode: {episode}, Step: {step}, Epsilon: {epsilon}, Reward: {reward}")

            if done or truncated:
                break

        epsilon *= epsilon_decay
    return q_table

In [44]:
from tqdm.auto import tqdm

def train_loop(episodes, lr, gamma, max_steps, epsilon, epsilon_decay, q_table):
    reward_list = []
    for episode in tqdm(range(episodes)):
        state, info = env.reset()

        total_reward = 0
        for step in range(max_steps):
            action = epsilon_greedy(epsilon, q_table, state)
            next_state, reward, done, truncated, info = env.step(action) 
            
            # Always update Q-table, even when episode ends
            q_table = update_q_table(q_table, state, action, reward, next_state, gamma, lr)
            state = next_state

            total_reward += reward
            if done or truncated:
                total_reward += reward 
                reward_list.append(total_reward)
                break
        
        epsilon *= epsilon_decay
    return q_table, reward_list

In [45]:
results = train_loop(n_training_steps, lr, gamma, max_steps, epsilon, epsilon_decay, q)

  0%|          | 0/100000 [00:00<?, ?it/s]

In [46]:
updated_q_table, reward_list = results
np.mean(reward_list), np.std(reward_list)

(np.float64(1.97826), np.float64(0.20738218920630574))

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="human")

def watch_agent_play(episodes=3):
    for episode in range(episodes):
        state, info = env.reset()
        done = False
        total_reward = 0
        
        print(f"\n=== Episode {episode + 1} ===")
        
        while not done:
            # Get action from trained Q-table (no exploration)
            action = greedy_policy(state, updated_q_table)
            
            # Take action
            next_state, reward, done, truncated, info = env.step(action)
            
            total_reward += reward
            state = next_state
            
            # The show the window 
            env.render()
            
            if done or truncated:
                break
    
    env.close()

watch_agent_play(episodes=3)


=== Episode 1 ===

=== Episode 2 ===

=== Episode 3 ===


: 

### Taxi-v3

In [21]:
import random 
import numpy as np 
import gymnasium as gym

env = gym.make("Taxi-v3", render_mode="rgb_array")

In [22]:
state_space = env.observation_space.n
action_space = env.action_space.n
print(f"State space: {state_space}")
print(f"Action space: {action_space}")

State space: 500
Action space: 6


In [23]:
def make_q_table(state_space, action_space):
    return np.zeros([state_space, action_space])

q_table = make_q_table(state_space, action_space)

In [29]:
def epsilon_greedy(epsilon, q_table, state):
    if epsilon > random.random():
        return random.randint(0, action_space - 1)
    else:
        return np.argmax(q_table[state])

In [32]:
from tqdm.auto import tqdm

def train_loop(epochs, q_table, epsilon, epsilon_decay, gamma, lr, max_steps):
    total_rewards = []

    for epoch in tqdm(range(epochs)):
        state, info = env.reset()

        for step in range(max_steps):
            move = epsilon_greedy(epsilon, q_table, state)
            next_state, reward, done, truncated, info = env.step(move)
            q_table[state, move] += lr * (reward + gamma * np.max(q_table[next_state]) - q_table[state, move])
            state = next_state
            if done or truncated:
                break
        total_rewards.append(reward)
        epsilon = max(epsilon * epsilon_decay, 0.01)
    return q_table, total_rewards

In [33]:
updated_q_table, reward_list = train_loop(epochs = 1000000, q_table = q_table, epsilon=1, epsilon_decay=0.99, gamma = 0.9, lr = 0.1, max_steps = 1000)

  0%|          | 0/1000000 [00:00<?, ?it/s]

In [34]:
np.mean(reward_list), np.std(reward_list)

(np.float64(19.999832), np.float64(0.05939673203131656))

In [39]:
env = gym.make("Taxi-v3", render_mode="human")

In [40]:
def visualize_loop(epochs, q_table):
    total_rewards = []

    for epoch in tqdm(range(epochs)):
        state, info = env.reset()

        while True:
            move = np.argmax(q_table[state])
            next_state, reward, done, truncated, info = env.step(move)
            state = next_state
            env.render()
            if done or truncated:
                break
        total_rewards.append(reward)

    env.close()
    return total_rewards

In [41]:
visualize_loop(5, updated_q_table)

  0%|          | 0/5 [00:00<?, ?it/s]

[20, 20, 20, 20, 20]